# MR Skeleton Structure Analysis

This notebook analyzes the **structural skeleton** of Metamorphic Relations (MRs), ignoring the specific features and focusing only on the relational patterns.

## Objective

Determine if AI-generated MRs follow the same **structural patterns** as human-designed ones.

## Skeleton Representation

Each MR is reduced to its skeleton:

```
Original: NS(m1) > NS(m2) => TTD(m2) >= TTD(m1) AND D(m2) >= D(m1)
Skeleton: [INC] => [INC AND INC]

Tree form:
MR
├── LHS
│   └── INC
└── RHS
    └── AND
        ├── INC
        └── INC
```

Where:
- **INC** = Increasing relation (>, >=)
- **DEC** = Decreasing relation (<, <=)
- **EQ** = Equality relation (=, ==)

## Metrics

- **Skeleton TED**: Tree Edit Distance on skeleton trees
- **Skeleton Pattern**: String representation of the pattern
- **Pattern Distribution**: How many MRs share each pattern

In [1]:
# ============================================================
# CONFIGURATION
# ============================================================

INPUT_FILES = [
    "mrs_AVs.txt",
    "mrs_DCs.txt",
]

OUTPUT_DIR = "output_skeleton"

# Visualization settings
HEATMAP_FIGSIZE = (14, 12)

print("Configuration loaded")

Configuration loaded


In [2]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Set, Tuple, Dict
from collections import Counter
from enum import Enum
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}/")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/opt/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/opt/anaconda3/lib/python3.12/site-

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.5 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Output directory: output_skeleton/


In [3]:
# ============================================================
# DATA MODEL
# ============================================================

class ComparisonOperator(Enum):
    GT = ">"
    GE = ">="
    LT = "<"
    LE = "<="
    EQ = "=="
    NE = "!="

class Direction(Enum):
    """Relational direction (the skeleton element)"""
    INC = "INC"   # Increasing: >, >=
    DEC = "DEC"   # Decreasing: <, <=
    EQ = "EQ"     # Equality: =, ==
    
    @classmethod
    def from_operator(cls, op: ComparisonOperator) -> 'Direction':
        if op in [ComparisonOperator.GT, ComparisonOperator.GE]:
            return cls.INC
        elif op in [ComparisonOperator.LT, ComparisonOperator.LE]:
            return cls.DEC
        else:
            return cls.EQ

@dataclass
class Atom:
    """Single comparison unit within an MR"""
    feature: str
    operator: ComparisonOperator
    
    @property
    def direction(self) -> Direction:
        return Direction.from_operator(self.operator)

@dataclass
class MetamorphicRelation:
    """Complete MR with LHS and RHS"""
    id: str
    group: str = "default"
    category: str = "normal"
    lhs_atoms: List[Atom] = field(default_factory=list)
    rhs_atoms: List[Atom] = field(default_factory=list)
    raw_text: str = ""
    
    @property
    def full_id(self) -> str:
        return f"{self.group}-{self.category}-{self.id}"

print("Data model ready")

Data model ready


In [4]:
# ============================================================
# SKELETON TREE REPRESENTATION
# ============================================================

@dataclass
class SkeletonNode:
    """Node in the skeleton tree (only structure, no features)"""
    label: str  # MR, LHS, RHS, AND, INC, DEC, EQ
    children: List['SkeletonNode'] = field(default_factory=list)
    
    def __repr__(self):
        if not self.children:
            return self.label
        return f"{self.label}({', '.join(repr(c) for c in self.children)})"
    
    def size(self) -> int:
        """Total nodes in tree"""
        return 1 + sum(c.size() for c in self.children)
    
    def to_pattern_string(self) -> str:
        """Convert to readable pattern string"""
        if self.label == "MR":
            lhs = self.children[0].to_pattern_string() if self.children else ""
            rhs = self.children[1].to_pattern_string() if len(self.children) > 1 else ""
            return f"{lhs} => {rhs}"
        elif self.label == "LHS" or self.label == "RHS":
            if self.children:
                return self.children[0].to_pattern_string()
            return ""
        elif self.label == "AND":
            parts = [c.to_pattern_string() for c in self.children]
            return " AND ".join(parts)
        else:
            return self.label

def mr_to_skeleton(mr: MetamorphicRelation) -> SkeletonNode:
    """
    Converts MR to skeleton tree (only directions, no features).
    
    Example:
        NS > ... => TTD >= AND D >=
        becomes:
        MR(LHS(INC), RHS(AND(INC, INC)))
    """
    def atoms_to_skeleton(atoms: List[Atom]) -> SkeletonNode:
        """Convert atoms to skeleton subtree"""
        if len(atoms) == 0:
            return SkeletonNode("EMPTY")
        elif len(atoms) == 1:
            return SkeletonNode(atoms[0].direction.value)
        else:
            # Multiple atoms: create AND node with direction children
            children = [SkeletonNode(a.direction.value) for a in atoms]
            return SkeletonNode("AND", children)
    
    lhs_skeleton = atoms_to_skeleton(mr.lhs_atoms)
    rhs_skeleton = atoms_to_skeleton(mr.rhs_atoms)
    
    lhs_node = SkeletonNode("LHS", [lhs_skeleton])
    rhs_node = SkeletonNode("RHS", [rhs_skeleton])
    
    return SkeletonNode("MR", [lhs_node, rhs_node])

def get_skeleton_pattern(mr: MetamorphicRelation) -> str:
    """Get readable pattern string for MR skeleton"""
    skeleton = mr_to_skeleton(mr)
    return skeleton.to_pattern_string()

def print_skeleton_tree(node: SkeletonNode, prefix: str = "", is_last: bool = True):
    """Pretty print skeleton tree"""
    connector = "└── " if is_last else "├── "
    print(prefix + connector + node.label)
    prefix += "    " if is_last else "│   "
    for i, child in enumerate(node.children):
        print_skeleton_tree(child, prefix, i == len(node.children) - 1)

print("Skeleton tree representation ready")

Skeleton tree representation ready


In [5]:
# ============================================================
# PARSER
# ============================================================

class MRParseError(Exception):
    pass

class MRParser:
    OPERATOR_MAP = {
        '>=': ComparisonOperator.GE, '<=': ComparisonOperator.LE,
        '>': ComparisonOperator.GT, '<': ComparisonOperator.LT,
        '==': ComparisonOperator.EQ, '!=': ComparisonOperator.NE,
        '=': ComparisonOperator.EQ,
    }
    
    def parse_atom(self, text):
        text = text.strip()
        match = re.search(r'(>=|<=|>|<|==|!=|=)', text)
        if not match:
            raise MRParseError(f"No operator found: {text}")
        
        op_str = match.group(1)
        operator = self.OPERATOR_MAP[op_str]
        left, right = text.split(op_str, 1)
        left = left.strip()
        
        # Extract feature name
        feat_match = re.match(r'(\w+)\s*\(', left)
        if feat_match:
            feature = feat_match.group(1)
        else:
            feature = left
        
        return Atom(feature=feature, operator=operator)
    
    def parse_mr(self, text, group='default', category='normal'):
        text = text.strip()
        text = re.sub(r"'?=>", "=>", text)
        
        mr_id = 'MR'
        id_match = re.match(r'^(MR\d*)\s*:\s*(.+)$', text)
        if id_match:
            mr_id = id_match.group(1)
            text = id_match.group(2)
        
        if '=>' not in text:
            raise MRParseError(f"Missing '=>': {text}")
        
        lhs, rhs = text.split('=>', 1)
        lhs_atoms = [self.parse_atom(p) for p in re.split(r'\s+AND\s+', lhs.strip(), flags=re.I) if p.strip()]
        rhs_atoms = [self.parse_atom(p) for p in re.split(r'\s+AND\s+', rhs.strip(), flags=re.I) if p.strip()]
        
        return MetamorphicRelation(id=mr_id, group=group, category=category,
                                   lhs_atoms=lhs_atoms, rhs_atoms=rhs_atoms, raw_text=text)

class MRFileLoader:
    def __init__(self):
        self.parser = MRParser()
    
    def load_txt(self, filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        mrs = []
        for i, line in enumerate(lines, 1):
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            try:
                if ',' in line.split('=>')[0]:
                    mr = self._parse_csv_line(line)
                else:
                    mr = self.parser.parse_mr(line)
                mrs.append(mr)
            except MRParseError as e:
                print(f"  Warning line {i}: {e}")
        return mrs
    
    def _parse_csv_line(self, line):
        line = line.replace("'=>", "=>")
        if '=>' not in line:
            raise MRParseError(f"Missing '=>': {line}")
        
        parts_by_arrow = line.split('=>', 1)
        left_part = parts_by_arrow[0].strip()
        rhs = parts_by_arrow[1].strip()
        
        comma_parts = left_part.split(',')
        if len(comma_parts) < 4:
            raise MRParseError(f"Invalid format: {line}")
        
        group = comma_parts[0].strip()
        mr_id = comma_parts[1].strip()
        category = comma_parts[2].strip()
        lhs = ' '.join(','.join(comma_parts[3:]).split())
        rhs = ' '.join(rhs.split())
        
        return self.parser.parse_mr(f"{mr_id}: {lhs} => {rhs}", group, category)

def load_mrs_from_file(filepath):
    return MRFileLoader().load_txt(filepath)

print("Parser ready")

Parser ready


In [6]:
# ============================================================
# SKELETON TREE EDIT DISTANCE (Zhang-Shasha)
# ============================================================

def tree_to_postorder(node: SkeletonNode) -> List[SkeletonNode]:
    """Get nodes in postorder"""
    result = []
    for child in node.children:
        result.extend(tree_to_postorder(child))
    result.append(node)
    return result

def leftmost_leaf_index(node: SkeletonNode, nodes: List[SkeletonNode], idx_map: Dict) -> int:
    """Find leftmost leaf index for Zhang-Shasha"""
    if not node.children:
        return idx_map[id(node)]
    return leftmost_leaf_index(node.children[0], nodes, idx_map)

def skeleton_ted(tree1: SkeletonNode, tree2: SkeletonNode) -> int:
    """
    Zhang-Shasha Tree Edit Distance for skeleton trees.
    Cost: insert=1, delete=1, rename=1 (if different labels)
    """
    nodes1 = tree_to_postorder(tree1)
    nodes2 = tree_to_postorder(tree2)
    
    n1, n2 = len(nodes1), len(nodes2)
    
    idx1 = {id(node): i for i, node in enumerate(nodes1)}
    idx2 = {id(node): i for i, node in enumerate(nodes2)}
    
    l1 = [leftmost_leaf_index(node, nodes1, idx1) for node in nodes1]
    l2 = [leftmost_leaf_index(node, nodes2, idx2) for node in nodes2]
    
    # Get keyroots
    def get_keyroots(nodes, l_list):
        kr = {}
        for i in range(len(nodes)):
            kr[l_list[i]] = i
        return sorted(kr.values())
    
    kr1 = get_keyroots(nodes1, l1)
    kr2 = get_keyroots(nodes2, l2)
    
    td = np.zeros((n1 + 1, n2 + 1))
    
    def cost(n1, n2):
        if n1 is None or n2 is None:
            return 1
        return 0 if n1.label == n2.label else 1
    
    for i in kr1:
        for j in kr2:
            m = i - l1[i] + 2
            n = j - l2[j] + 2
            fd = np.zeros((m, n))
            
            for x in range(1, m):
                fd[x, 0] = fd[x-1, 0] + 1
            for y in range(1, n):
                fd[0, y] = fd[0, y-1] + 1
            
            for x in range(1, m):
                for y in range(1, n):
                    i1 = l1[i] + x - 1
                    j1 = l2[j] + y - 1
                    
                    if l1[i1] == l1[i] and l2[j1] == l2[j]:
                        fd[x, y] = min(
                            fd[x-1, y] + 1,
                            fd[x, y-1] + 1,
                            fd[x-1, y-1] + cost(nodes1[i1], nodes2[j1])
                        )
                        td[i1+1, j1+1] = fd[x, y]
                    else:
                        p = l1[i1] - l1[i]
                        q = l2[j1] - l2[j]
                        fd[x, y] = min(
                            fd[x-1, y] + 1,
                            fd[x, y-1] + 1,
                            fd[p, q] + td[i1+1, j1+1]
                        )
    
    return int(td[n1, n2])

def compute_skeleton_ted_matrix(mrs: List[MetamorphicRelation]) -> Tuple[pd.DataFrame, Dict]:
    """
    Computes pairwise Skeleton TED distance matrix.
    Returns: (distance_matrix, skeletons_dict)
    """
    n = len(mrs)
    mr_ids = [mr.full_id for mr in mrs]
    
    # Build skeleton trees
    skeletons = {mr.full_id: mr_to_skeleton(mr) for mr in mrs}
    
    dist_matrix = np.zeros((n, n), dtype=int)
    
    for i in range(n):
        for j in range(i+1, n):
            dist = skeleton_ted(skeletons[mr_ids[i]], skeletons[mr_ids[j]])
            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist
    
    return (
        pd.DataFrame(dist_matrix, index=mr_ids, columns=mr_ids),
        skeletons
    )

print("Skeleton TED ready")

Skeleton TED ready


In [7]:
# ============================================================
# PATTERN ANALYSIS
# ============================================================

def analyze_skeleton_patterns(mrs: List[MetamorphicRelation]) -> pd.DataFrame:
    """
    Analyzes distribution of skeleton patterns.
    """
    patterns = []
    for mr in mrs:
        pattern = get_skeleton_pattern(mr)
        skeleton = mr_to_skeleton(mr)
        patterns.append({
            'MR_ID': mr.full_id,
            'Pattern': pattern,
            'Tree_Size': skeleton.size(),
            'N_LHS': len(mr.lhs_atoms),
            'N_RHS': len(mr.rhs_atoms),
            'LHS_Dirs': '+'.join(a.direction.value for a in mr.lhs_atoms),
            'RHS_Dirs': '+'.join(a.direction.value for a in mr.rhs_atoms),
        })
    
    return pd.DataFrame(patterns)

def get_pattern_distribution(pattern_df: pd.DataFrame) -> pd.DataFrame:
    """Get distribution of unique patterns"""
    counts = pattern_df['Pattern'].value_counts().reset_index()
    counts.columns = ['Pattern', 'Count']
    counts['Percentage'] = 100 * counts['Count'] / len(pattern_df)
    return counts

def analyze_direction_combinations(pattern_df: pd.DataFrame) -> pd.DataFrame:
    """Analyze LHS->RHS direction combinations"""
    combos = pattern_df.groupby(['LHS_Dirs', 'RHS_Dirs']).size().reset_index(name='Count')
    combos['Percentage'] = 100 * combos['Count'] / len(pattern_df)
    combos = combos.sort_values('Count', ascending=False)
    return combos

print("Pattern analysis ready")

Pattern analysis ready


In [8]:
# ============================================================
# MATRIX ANALYSIS
# ============================================================

def analyze_ted_matrix(dist_matrix: pd.DataFrame, name: str) -> Dict:
    """Analyze TED distance matrix"""
    n = len(dist_matrix)
    upper_tri = np.triu_indices(n, k=1)
    
    distances = dist_matrix.values[upper_tri]
    
    lines = []
    lines.append("=" * 60)
    lines.append(f"SKELETON TED ANALYSIS: {name}")
    lines.append("=" * 60)
    
    lines.append("\n--- Distance Statistics ---")
    lines.append(f"Total pairs: {len(distances)}")
    lines.append(f"Mean distance: {np.mean(distances):.2f}")
    lines.append(f"Median distance: {np.median(distances):.2f}")
    lines.append(f"Min: {np.min(distances):.0f} | Max: {np.max(distances):.0f}")
    lines.append(f"Std: {np.std(distances):.2f}")
    
    lines.append("\n--- Distance Distribution ---")
    unique_dists = sorted(set(distances))
    for d in unique_dists:
        count = np.sum(distances == d)
        pct = 100 * count / len(distances)
        lines.append(f"  TED = {d:.0f}: {count:3d} pairs ({pct:5.1f}%)")
    
    lines.append(f"\nPairs with identical structure (TED=0): {np.sum(distances == 0)}")
    
    # Identical pairs
    identical_pairs = [(dist_matrix.index[i], dist_matrix.columns[j]) 
                       for i in range(n) for j in range(i+1, n) 
                       if dist_matrix.iloc[i, j] == 0]
    
    if identical_pairs:
        lines.append("\n--- Identical Structure Pairs (TED=0) ---")
        for a, b in identical_pairs:
            lines.append(f"  {a} <-> {b}")
    
    for line in lines:
        print(line)
    
    return {
        'mean_dist': np.mean(distances),
        'n_identical': len(identical_pairs),
        'identical_pairs': identical_pairs,
        'report_lines': lines
    }

print("Matrix analysis ready")

Matrix analysis ready


In [9]:
# ============================================================
# PDF GENERATION
# ============================================================

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform
import matplotlib.colors as mcolors
from matplotlib.patches import Patch

# Discrete blue palette for distance levels in heatmap
DISTANCE_COLORS = [
    '#F0F0F0',  # 0 - almost white (identical)
    '#C6DBEF',  # 1
    '#9ECAE1',  # 2
    '#6BAED6',  # 3
    '#3182BD',  # 4
    '#08519C',  # 5
    '#063E78',  # 6
    '#042F5A',  # 7
    '#021E3C',  # 8+
]

# Cluster colors for dendrogram color strip
CLUSTER_COLORS = [
    '#E41A1C', '#377EB8', '#4DAF4A', '#984EA3',
    '#FF7F00', '#A65628', '#F781BF', '#66C2A5',
    '#FC8D62', '#8DA0CB', '#E78AC3', '#A6D854',
]

def generate_distance_heatmap_pdf(dist_matrix: pd.DataFrame, output_path: str, title: str):
    """
    Generate discrete-color clustermap (heatmap + dendrogram) for TED distance matrix.
    Uses seaborn.clustermap for correct alignment between dendrogram and heatmap.
    - Discrete blue palette for distance values
    - Cluster color strip (n_clusters = n_unique_distances)
    - Numeric values in each cell with adaptive text color
    """
    matrix = dist_matrix.values.astype(int)
    max_dist = int(matrix.max())
    
    # Build discrete colormap
    n_colors = min(max_dist + 1, len(DISTANCE_COLORS))
    colors = DISTANCE_COLORS[:n_colors]
    cmap = mcolors.ListedColormap(colors)
    bounds = list(range(n_colors + 1))
    norm = mcolors.BoundaryNorm(bounds, cmap.N)
    
    # Hierarchical clustering
    condensed = squareform(matrix.astype(float))
    Z = linkage(condensed, method='average')
    
    # Clusters: one per unique distance level
    unique_dists = sorted(np.unique(matrix))
    n_clusters = min(len(unique_dists), len(CLUSTER_COLORS))
    clusters = fcluster(Z, t=n_clusters, criterion='maxclust')
    
    # Cluster color strip
    cluster_color_map = {i+1: CLUSTER_COLORS[i] for i in range(n_clusters)}
    row_colors = pd.Series(clusters, index=dist_matrix.index).map(cluster_color_map)
    
    # Clustermap (handles dendrogram-heatmap alignment automatically)
    g = sns.clustermap(
        dist_matrix,
        method='average',
        metric='precomputed',
        row_linkage=Z,
        col_linkage=Z,
        cmap=cmap,
        norm=norm,
        linewidths=0.5,
        linecolor='white',
        figsize=HEATMAP_FIGSIZE,
        annot=True,
        fmt='d',
        annot_kws={'size': max(6, min(10, 120 // len(matrix))), 'weight': 'bold'},
        row_colors=row_colors,
        col_colors=row_colors,
        dendrogram_ratio=(0.12, 0.12),
        cbar_pos=None,
        tree_kws={'linewidths': 1.5},
    )
    
    # Fix annotation text colors based on cell background
    for text in g.ax_heatmap.texts:
        val = int(text.get_text())
        color_idx = min(val, len(colors) - 1)
        rgb = mcolors.to_rgb(colors[color_idx])
        brightness = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]
        text.set_color('white' if brightness < 0.5 else '#333333')
    
    # Rotate bottom labels for readability
    g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(),
                                  rotation=45, ha='right', fontsize=8)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)
    
    # Title
    g.fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    
    # Custom legend (distances + clusters)
    legend_elements = []
    for d in unique_dists:
        color_idx = min(int(d), len(colors) - 1)
        legend_elements.append(Patch(facecolor=colors[color_idx], edgecolor='gray',
                                     label=f'Distance {int(d)}'))
    legend_elements.append(Patch(facecolor='none', edgecolor='none', label=''))
    for c in range(1, n_clusters + 1):
        legend_elements.append(Patch(facecolor=CLUSTER_COLORS[c - 1], edgecolor='gray',
                                     label=f'Cluster {c}'))
    g.ax_heatmap.legend(handles=legend_elements, loc='center left',
                         bbox_to_anchor=(1.15, 0.5), frameon=True, fontsize=8,
                         title='Legend', title_fontsize=9)
    
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {output_path}")

def generate_pattern_distribution_pdf(pattern_dist: pd.DataFrame, output_path: str, title: str):
    """Generate pattern distribution chart"""
    fig, ax = plt.subplots(figsize=(12, max(6, len(pattern_dist) * 0.5)))
    
    y_pos = range(len(pattern_dist))
    bars = ax.barh(y_pos, pattern_dist['Count'], color='steelblue', alpha=0.8)
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(pattern_dist['Pattern'], fontsize=9)
    ax.set_xlabel('Count')
    ax.set_title(f'Skeleton Pattern Distribution\n{title}', fontweight='bold')
    
    for i, (count, pct) in enumerate(zip(pattern_dist['Count'], pattern_dist['Percentage'])):
        ax.text(count + 0.1, i, f'{count} ({pct:.1f}%)', va='center', fontsize=9)
    
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {output_path}")

def generate_direction_combo_pdf(combo_df: pd.DataFrame, output_path: str, title: str):
    """Generate direction combination heatmap"""
    # Pivot for heatmap
    pivot = combo_df.pivot_table(index='LHS_Dirs', columns='RHS_Dirs', 
                                  values='Count', fill_value=0)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(pivot, annot=True, fmt='.0f', cmap='Blues', 
                square=True, linewidths=0.5, ax=ax,
                cbar_kws={'shrink': 0.8, 'label': 'Count'})
    
    ax.set_title(f'LHS → RHS Direction Combinations\n{title}', fontweight='bold')
    ax.set_xlabel('RHS Directions')
    ax.set_ylabel('LHS Directions')
    
    plt.tight_layout()
    plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {output_path}")

print("PDF generators ready")

PDF generators ready


In [10]:
# ============================================================
# REPORT GENERATION
# ============================================================

def generate_skeleton_report(filename: str, mrs: List, pattern_df: pd.DataFrame,
                              pattern_dist: pd.DataFrame, combo_df: pd.DataFrame,
                              analysis: Dict, skeletons: Dict, output_path: str):
    """Generate comprehensive skeleton analysis report"""
    lines = []
    lines.append("=" * 70)
    lines.append(f"SKELETON STRUCTURE ANALYSIS REPORT: {filename}")
    lines.append("=" * 70)
    lines.append("")
    lines.append("This report analyzes the STRUCTURAL SKELETON of MRs.")
    lines.append("Features are ignored; only relational patterns (INC/DEC/EQ) are compared.")
    lines.append("")
    
    # Summary
    lines.append("-" * 50)
    lines.append("SUMMARY")
    lines.append("-" * 50)
    lines.append(f"Total MRs: {len(mrs)}")
    lines.append(f"Unique skeleton patterns: {len(pattern_dist)}")
    lines.append(f"Pairs with identical structure: {analysis['n_identical']}")
    lines.append(f"Mean skeleton TED: {analysis['mean_dist']:.2f}")
    lines.append("")
    
    # Pattern distribution
    lines.append("-" * 50)
    lines.append("SKELETON PATTERN DISTRIBUTION")
    lines.append("-" * 50)
    lines.append(pattern_dist.to_string(index=False))
    lines.append("")
    
    # Direction combinations
    lines.append("-" * 50)
    lines.append("LHS → RHS DIRECTION COMBINATIONS")
    lines.append("-" * 50)
    lines.append(combo_df.to_string(index=False))
    lines.append("")
    
    # Individual MR skeletons
    lines.append("-" * 50)
    lines.append("MR SKELETON PATTERNS")
    lines.append("-" * 50)
    for _, row in pattern_df.iterrows():
        lines.append(f"\n{row['MR_ID']}:")
        lines.append(f"  Pattern: {row['Pattern']}")
        lines.append(f"  LHS: [{row['LHS_Dirs']}] ({row['N_LHS']} atoms)")
        lines.append(f"  RHS: [{row['RHS_Dirs']}] ({row['N_RHS']} atoms)")
    lines.append("")
    
    # TED analysis
    lines.append("\n")
    lines.extend(analysis['report_lines'])
    
    # Interpretation
    lines.append("\n")
    lines.append("-" * 50)
    lines.append("INTERPRETATION")
    lines.append("-" * 50)
    lines.append("")
    lines.append("Skeleton TED measures structural similarity:")
    lines.append("  TED = 0: Identical relational pattern")
    lines.append("  TED = 1: One direction differs (e.g., INC vs DEC)")
    lines.append("  TED = 2+: Multiple structural differences")
    lines.append("")
    lines.append("Direction codes:")
    lines.append("  INC = Increasing (>, >=)")
    lines.append("  DEC = Decreasing (<, <=)")
    lines.append("  EQ  = Equality (=, ==)")
    
    lines.append("\n" + "=" * 70)
    lines.append("END OF REPORT")
    lines.append("=" * 70)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    
    print(f"  Saved: {output_path}")

print("Report generator ready")

Report generator ready


In [11]:
# ============================================================
# MAIN PROCESSING
# ============================================================

def process_mr_file(filepath: str, output_dir: str) -> Dict:
    """Process a single MR file"""
    filename = Path(filepath).stem
    print(f"\n{'='*60}")
    print(f"Processing: {filepath}")
    print(f"{'='*60}")
    
    # Load MRs
    mrs = load_mrs_from_file(filepath)
    print(f"MRs loaded: {len(mrs)}")
    
    if len(mrs) == 0:
        return {'filename': filename, 'status': 'empty'}
    
    # Remove duplicates
    unique_mrs = []
    seen = set()
    for mr in mrs:
        if mr.full_id not in seen:
            unique_mrs.append(mr)
            seen.add(mr.full_id)
    mrs = unique_mrs
    print(f"Unique MRs: {len(mrs)}")
    
    # Show skeleton examples
    print("\n--- Skeleton Examples ---")
    for mr in mrs[:3]:
        print(f"\n{mr.full_id}:")
        print(f"  Original: {mr.raw_text[:60]}...")
        print(f"  Skeleton: {get_skeleton_pattern(mr)}")
    
    # Analyze patterns
    print("\nAnalyzing skeleton patterns...")
    pattern_df = analyze_skeleton_patterns(mrs)
    pattern_dist = get_pattern_distribution(pattern_df)
    combo_df = analyze_direction_combinations(pattern_df)
    
    print(f"Unique patterns: {len(pattern_dist)}")
    
    # Compute TED
    print("\nComputing Skeleton TED...")
    dist_matrix, skeletons = compute_skeleton_ted_matrix(mrs)
    
    # Analysis
    print("")
    analysis = analyze_ted_matrix(dist_matrix, filename)
    
    # Generate PDFs
    print("\nGenerating PDFs...")
    
    # 1. Skeleton TED distance heatmap with dendrogram
    generate_distance_heatmap_pdf(
        dist_matrix,
        os.path.join(output_dir, f"{filename}_skeleton_ted_distance.pdf"),
        f"Skeleton TED Distance\n{filename}"
    )
    
    # 2. Pattern distribution
    generate_pattern_distribution_pdf(
        pattern_dist,
        os.path.join(output_dir, f"{filename}_skeleton_patterns.pdf"),
        filename
    )
    
    # 3. Direction combinations
    generate_direction_combo_pdf(
        combo_df,
        os.path.join(output_dir, f"{filename}_direction_combos.pdf"),
        filename
    )
    
    # Generate report
    generate_skeleton_report(
        filename, mrs, pattern_df, pattern_dist, combo_df,
        analysis, skeletons,
        os.path.join(output_dir, f"{filename}_skeleton_report.txt")
    )
    
    # Export matrices
    dist_matrix.to_csv(os.path.join(output_dir, f"{filename}_skeleton_ted_matrix.csv"))
    pattern_df.to_csv(os.path.join(output_dir, f"{filename}_skeleton_patterns.csv"), index=False)
    
    print(f"\nResult: {len(mrs)} MRs → {len(pattern_dist)} unique skeleton patterns")
    
    return {
        'filename': filename,
        'n_mrs': len(mrs),
        'n_patterns': len(pattern_dist),
        'n_identical_pairs': analysis['n_identical'],
        'mean_ted': analysis['mean_dist'],
        'status': 'success'
    }

print("Main processing ready")

Main processing ready


In [12]:
# ============================================================
# EXECUTION
# ============================================================

print("=" * 60)
print("MR SKELETON STRUCTURE ANALYSIS")
print("(Features ignored, only relational patterns compared)")
print("=" * 60)

results = []
for fp in INPUT_FILES:
    if os.path.exists(fp):
        results.append(process_mr_file(fp, OUTPUT_DIR))
    else:
        print(f"\nFile not found: {fp}")

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
for r in results:
    if r.get('status') == 'success':
        print(f"\n{r['filename']}:")
        print(f"  MRs: {r['n_mrs']}")
        print(f"  Unique skeleton patterns: {r['n_patterns']}")
        print(f"  Pairs with identical skeleton: {r['n_identical_pairs']}")
        print(f"  Mean Skeleton TED: {r['mean_ted']:.2f}")

MR SKELETON STRUCTURE ANALYSIS
(Features ignored, only relational patterns compared)

Processing: mrs_AVs.txt
MRs loaded: 20
Unique MRs: 17

--- Skeleton Examples ---

base-normal-MR1:
  Original: NS(m1) > NS(m2) => TTD(m2) >= TTD(m1) AND D(m2) >= D(m1)...
  Skeleton: INC => INC AND INC

base-normal-MR2:
  Original: OC(m1) > OC(m2) => TTD(m1) >= TTD(m2) AND D(m1) >= D(m2)...
  Skeleton: INC => INC AND INC

base-normal-MR3:
  Original: WPC(m1) > WPC(m2) => TTD(m1) >= TTD(m2) AND D(m1) >= D(m2)...
  Skeleton: INC => INC AND INC

Analyzing skeleton patterns...
Unique patterns: 6

Computing Skeleton TED...

SKELETON TED ANALYSIS: mrs_AVs

--- Distance Statistics ---
Total pairs: 136
Mean distance: 1.90
Median distance: 2.00
Min: 0 | Max: 4
Std: 1.17

--- Distance Distribution ---
  TED = 0:  28 pairs ( 20.6%)
  TED = 1:  10 pairs (  7.4%)
  TED = 2:  51 pairs ( 37.5%)
  TED = 3:  41 pairs ( 30.1%)
  TED = 4:   6 pairs (  4.4%)

Pairs with identical structure (TED=0): 28

--- Identical Stru

---

## Output Files

| File | Description |
|------|-------------|
| `*_skeleton_ted_distance.pdf` | TED distance heatmap with dendrogram (discrete blue palette) |
| `*_skeleton_patterns.pdf` | Distribution of unique patterns |
| `*_direction_combos.pdf` | LHS→RHS direction combinations |
| `*_skeleton_report.txt` | Full analysis report |
| `*_skeleton_ted_matrix.csv` | TED distance matrix as CSV |
| `*_skeleton_patterns.csv` | All MRs with their skeleton patterns |

## Skeleton Representation

```
Original MR:  NS(m1) > NS(m2) => TTD(m2) >= TTD(m1) AND D(m2) >= D(m1)
Skeleton:     INC => INC AND INC

Tree:
MR
├── LHS
│   └── INC
└── RHS
    └── AND
        ├── INC
        └── INC
```

## Interpretation

- **TED = 0**: MRs have identical relational structure
- **TED = 1**: One direction differs (e.g., INC vs DEC)
- **TED = 2+**: Multiple structural differences

If AI-generated MRs have low TED compared to human MRs, they follow similar structural patterns.